1 Aqui realizamos el analisis de los datos y tanamos una pequeña observacion en la estuctura de datos, que observaremos a traves de los distintos frameworks cual es el estado actual y la calidad de los datos facilitados

In [ ]:
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
from google.colab import files
import os

drive.mount('/content/drive')
df.to_csv("cleaned_dataset.csv", index=False)

files.download("cleaned_dataset.csv")

df = pd.read_csv("/content/drive/MyDrive/greenmart_customers_products.csv")
print(f'El indice es el siguiente {df.columns}\nTenemos unos {len(df)} datos en el csv\nEstos datos estan clasificados con estos tipos de datos \n{df.dtypes}\nTenemos datos perdidos en cada columna equivalentes a \n{df.isnull().sum()}\nAdemas de contar las rows duplicadas {df.duplicated().sum()}')
print(f'Los datos estadisticos son: \n{df.describe().round(2)}')

Mounted at /content/drive
El indice es el siguiente Index(['customer_id', 'customer_name', 'age', 'city', 'product_id',
       'product_name', 'category', 'purchase_date', 'purchase_quantity',
       'price_per_unit', 'total_spent'],
      dtype='object')
Tenemos unos 10400 datos en el csv
Estos datos estan clasificados con estos tipos de datos 
customer_id            int64
customer_name         object
age                  float64
city                  object
product_id             int64
product_name          object
category              object
purchase_date         object
purchase_quantity    float64
price_per_unit       float64
total_spent          float64
dtype: object
Tenemos datos perdidos en cada columna equivalentes a 
customer_id            0
customer_name        722
age                  722
city                 722
product_id             0
product_name           0
category               0
purchase_date        722
purchase_quantity    708
price_per_unit         0
total_spent   

2 En esta segunda parte del trabajo de limpieza vamos a analizar un poco mas profundo ciertos aspectos que aun no se han trabajado para ser mas minuciosos con los datos facilitados en el CSV y mejorar la calidad de los datos.

In [ ]:
print(f'sabemos que los duplicados completos, es decir de toda la fila e informacion son {df.duplicated().sum()} dentro de todo el frameworks es decir que entorno al {((df.duplicated().sum()/len(df))*100).round(2)}% son datos duplicados')
print(f'Podemos ver que las 3 categorias de informacion de clientes "customer_name","age" y "city" tienen las mismas perdidas de datos {df["customer_name"].isna().sum()}')
print(f'Obsevamos tambien que la perdida de datos de clientes es la misma que la de la fecha de compra o "purchase_date" que son {df["purchase_date"].isna().sum()} fechas de compra mal registradas o perdidas')
print(f'En cambio tenemos unos datos no registrados en el "total_spent" y en el "purchase_quantity" que ambos coinciden en la misma cantidad de informacion perdida {df["total_spent"].isna().sum()}')
print(f'Creemos un error de datos en el "purchase_dates, debido al tipo de datos del mismo "{df["purchase_date"].dtype}" y quizas lo deberiamos de convertir a "datetime"')
invalid_dates = pd.to_datetime(df["purchase_date"], errors="coerce").isna().sum()
print(f'Se detectan {invalid_dates} fechas inválidas o imposibles de convertir a datetime')
print(f'Parece que hay un error de tipo tambien en los datos "age" que estan en "{df["age"].dtype}" y quizas es mas util tenerlos en "int"')
print(f'Hay otro valor de "purchase_quantity". que esta en {df['purchase_quantity'].dtype} y deberia de estar en "int')
print(f'El resto de columnas con tipo de datos "object" seria bueno cambiarlos a "STRING".')
print(f'En cuanto a Valores inusuales tenemos varios en mente empezamos por "total_spent" donde gracias a el .describe() del apartado anterior vemos que la media de compra es {df['total_spent'].mean().round(2)}\nvemos una compra exagerada en el max por valor a {round(df['total_spent'].max(),2)}')
print(f'Otros valores inusuales pueden ser en "price_per_unit" donde vemos un precio maximo muy alejado a la media({round(df['price_per_unit'].mean(),2)}) y el maximo ({round(df['price_per_unit'].max(),2)})')
print(f'Al igual que los valores de "purchase_quantity" que son valores igaulemnte llamativos. Media({round(df['purchase_quantity'].mean(),2)}) y el maximo ({round(df['purchase_quantity'].max(),2)})')
print(f'Los id de clientes duplicados abarcan {df["customer_id"].duplicated().sum()} serian los clientes recurrentes y los duplicados de id de producto abacan {df["product_id"].duplicated().sum()}')


sabemos que los duplicados completos, es decir de toda la fila e informacion son 367 dentro de todo el frameworks es decir que entorno al 3.53% son datos duplicados
Podemos ver que las 3 categorias de informacion de clientes "customer_name","age" y "city" tienen las mismas perdidas de datos 722
Obsevamos tambien que la perdida de datos de clientes es la misma que la de la fecha de compra o "purchase_date" que son 722 fechas de compra mal registradas o perdidas
En cambio tenemos unos datos no registrados en el "total_spent" y en el "purchase_quantity" que ambos coinciden en la misma cantidad de informacion perdida 708
Creemos un error de datos en el "purchase_dates, debido al tipo de datos del mismo "object" y quizas lo deberiamos de convertir a "datetime"
Se detectan 722 fechas inválidas o imposibles de convertir a datetime
Parece que hay un error de tipo tambien en los datos "age" que estan en "float64" y quizas es mas util tenerlos en "int"
Hay otro valor de "purchase_quantity". que 

3.Define a clear cleaning strategy before applying transformations, documenting whether each issue will be removed, corrected, imputed, standardized, or flagged for review.

Se aplicará una estrategia de limpieza basada en preservar la mayor cantidad de información útil sin introducir sesgos innecesarios. Los registros duplicados exactos serán eliminados, ya que representan redundancia en la captura de datos. Los valores faltantes en información de cliente (nombre, edad y ciudad) no se eliminarán de forma automática, ya que pueden corresponder a compras válidas en un entorno retail, pero serán marcados para revisión cuando afecten análisis demográficos. Los valores ausentes en purchase_date, purchase_quantity y total_spent se considerarán críticos cuando aparecen conjuntamente, por lo que dichos registros serán eliminados al no ser posible su reconstrucción fiable. Los tipos de datos serán estandarizados, convirtiendo fechas a formato datetime y variables numéricas a tipos adecuados. Finalmente, los valores inusuales en gasto, cantidad y precio unitario serán identificados y marcados para análisis posterior, evitando su eliminación inmediata hasta confirmar si representan errores o transacciones reales.


4 Remove exact duplicate rows and verify the number of rows before and after deduplication.

In [ ]:
print(f'el numero de registros antes de realizar el borrado de duplicados criticos es {len(df)}')
print(f'El numero de registros duplicados es {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'el numero de registros despues de realizar el borrado de duplicados criticos es {len(df)}')

el numero de registros antes de realizar el borrado de duplicados criticos es 9692
El numero de registros duplicados es 352
el numero de registros despues de realizar el borrado de duplicados criticos es 9340


5 Standardize column data types by converting purchase dates to a valid date format, customer and product identifiers to stable identifier fields, and numeric purchase fields to appropriate numeric types.

In [ ]:
print(df.dtypes)
string = ["customer_id","customer_name","city","product_name","category"]
df[string] = df[string].astype("string")
df["purchase_date"] = pd.to_datetime(df["purchase_date"])
enteros = ["age","purchase_quantity"]
df["age"] = df["age"].replace([np.inf, -np.inf], np.nan)
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].astype("Int64")
df["purchase_quantity"] = df["purchase_quantity"].astype("Int64")
print(df.dtypes)



customer_id          string[python]
customer_name        string[python]
age                           Int64
city                 string[python]
product_id                    int64
product_name         string[python]
category             string[python]
purchase_date                object
purchase_quantity             Int64
price_per_unit              float64
total_spent                 float64
dtype: object
customer_id          string[python]
customer_name        string[python]
age                           Int64
city                 string[python]
product_id                    int64
product_name         string[python]
category             string[python]
purchase_date        datetime64[ns]
purchase_quantity             Int64
price_per_unit              float64
total_spent                 float64
dtype: object


6 Handle missing values in customer, city, age, purchase date, purchase quantity, and total spending fields using a documented and reproducible approach suitable for the dataset.

Los valores faltantes se gestionan mediante un enfoque basado en reglas, dependiendo de la naturaleza de cada variable y su impacto en el análisis.

Los campos relacionados con el cliente (customer_name, city) no se eliminan, ya que la ausencia de estos datos puede deberse a compras válidas con registro incompleto. En su lugar, estos registros se mantienen y pueden ser marcados para análisis posterior.

Los valores faltantes en purchase_date se pueden considerar como un error de integracion que suceda en retail y al no realizar un analisis temporar puro podemos considerar que soin datos significativos.

En las variables transaccionales (purchase_quantity, total_spent), se eliminan los registros donde ambas variables son nulas, debido a que no es posible reconstruir la información de la compra. Si falta solo una de ellas, podría considerarse imputación basada en la relación entre variables, pero en este caso se prioriza la integridad de los datos eliminando registros incompletos críticos.

La variable age se mantiene en formato entero nullable para permitir valores faltantes sin introducir sesgos, como hemos realizado en el apartado anterior.

Todo el proceso se implementa de forma reproducible utilizando operaciones de pandas.

In [ ]:
df["missing_customer_info"] = df[["customer_name", "city"]].isna().any(axis=1)
df["missing_date_flag"] = df["purchase_date"].isna()
df = df.dropna(subset=["purchase_quantity", "total_spent"], how="all")

7.Validate the calculation of `total_spent` against `purchase_quantity * price_per_unit` and correct or confirm totals using a tolerance suitable for currency rounding.


In [ ]:
df["expected"] = df["purchase_quantity"] * df["price_per_unit"]
df["check"] = np.isclose(df["total_spent"],df["expected"],atol = 0.01)
print(f'Errores de chequeo \n{(-df["check"]).sum()}')
df = df.dropna(subset=["purchase_quantity", "total_spent"])
df = df.drop(columns=["expected"])


Errores de chequeo 
0


8 Detect and review outliers in `age`, `purchase_quantity`, `price_per_unit`, and `total_spent`, applying a documented rule such as range validation, interquartile range analysis, or business-rule-based flagging.

In [ ]:
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return ~df[column].between(limite_inferior, limite_superior)

df["outlier_age"] = ~df["age"].between(0, 100)
print(f'Outliers de age  = {(df["outlier_age"]).sum()} esta cantidad es de personas menores de 0 años y mayores de 100')
df["outlier_purchase_quantity"] = detect_outliers(df, "purchase_quantity")
df["outlier_price_per_unit"] = detect_outliers(df, "price_per_unit")
df["outlier_total_spent"] = detect_outliers(df, "total_spent")
print(f'Outliers purchase = {df["outlier_purchase_quantity"].sum()}')
print(f'Outliers price = {df["outlier_price_per_unit"].sum()}')
print(f'Outliers total spent = {df["outlier_total_spent"].sum()}')


Outliers de age  = 722 esta cantidad es de personas menores de 0 años y mayores de 100
Outliers purchase = 808
Outliers price = 100
Outliers total spent = 903


9 Normalize categorical and text fields where applicable, including customer names, cities, product names, and product categories, while preserving the original business meaning.


In [ ]:
cols = ["customer_name","city","product_name","category"]
for col in cols:
    df[col] = (df[col].astype("string").str.lower().str.strip())
print("Unificamos criterios todo minuscula sin espacios y en tipo de datos STRING")

Unificamos criterios todo minuscula sin espacios y en tipo de datos STRING


 10 Export the cleaned dataset as a CSV file and include final validation checks confirming that the output is ready for future analysis.

In [ ]:
df.to_csv("/content/drive/MyDrive/cleaned_dataset.csv", index=False)

# VALIDACIÓN FINAL

print("Valores nulos por columna:")
print(df.isna().sum())

print("\nTipos de datos:")
print(df.dtypes)

print("\nDuplicados:")
print(df.duplicated().sum())

print("\nFilas finales:")
print(len(df))

print("\nOutliers detectados:")
print(df[[
    "outlier_age",
    "outlier_purchase_quantity",
    "outlier_price_per_unit",
    "outlier_total_spent"
]].sum())

Valores nulos por columna:
customer_id                   0
customer_name                14
age                          14
city                         14
product_id                    0
product_name                  0
category                      0
purchase_date                14
purchase_quantity             0
price_per_unit                0
total_spent                   0
outlier_age                   0
outlier_purchase_quantity     0
outlier_price_per_unit        0
outlier_total_spent           0
missing_customer_info         0
missing_date_flag             0
check                         0
dtype: int64

Tipos de datos:
customer_id                           int64
customer_name                string[python]
age                                 float64
city                         string[python]
product_id                            int64
product_name                 string[python]
category                     string[python]
purchase_date                        object
purchase_quanti

Nombres que faltan en los datos de CSV, quizas clientes no fidelizados o registrados 722
esta es la clasificacion total de categorias por productos, en lso cuales se detectan muchos duplicados, es decir los
productos contienen la misma clase pero con todo minuscula o mayuscula y los productos pueden ser igual se ha sometido a una 
homogeneizacion de nombres y categorias y lo hemos podrido reducir a a 6 productos con 4 categorias sin perdidda de informacion
      product_name  numero_categorias
0   almond butter                  4
1      fresh milk                  4
2         granola                  4
3            kale                  4
4  organic apples                  4
5            tofu                  4
product_name   purchase_quantity
almond butter  101.0                1
               108.0                1
               140.0                2
               153.0                1
               156.0                1
                                   ..
tofu           179